# 02 — Churn Model Training + SHAP + AI Retention Messages
### E-commerce Customer Churn Prediction — ML Pipeline (Part 2/2)

**Pipeline stages covered here:** `ML Churn Prediction -> SHAP Reasons -> AI Retention Message -> (artifacts for the dashboard)`

This notebook:
1. Loads `churn_with_sentiment.csv` produced by Notebook 1.
2. Encodes features and trains **XGBoost** and **Random Forest** classifiers, compares them.
3. Runs **SHAP** on the best model to get global feature importance and per-customer churn reasons (e.g. "negative review", "fewer orders", "high returns/complaints").
4. Flags **high-risk** customers and uses **Gemini** to auto-generate a short, personalised **retention SMS + discount offer** referencing their top churn reasons.
5. Saves all artifacts (models, encoders, feature list, SHAP explainer, final results table) to Google Drive — these are what the FastAPI backend will load later.

## 1. Setup

In [ ]:
!pip install -q xgboost shap scikit-learn imbalanced-learn google-genai joblib

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
SAVE_DIR = "/content/drive/MyDrive/churn_prediction_project"
os.makedirs(SAVE_DIR, exist_ok=True)

DATA_PATH = os.path.join(SAVE_DIR, "churn_with_sentiment.csv")
print("Loading:", DATA_PATH)

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

## 2. Feature engineering

In [ ]:
from sklearn.preprocessing import LabelEncoder

CATEGORICAL_COLS = [
    "PreferredLoginDevice", "PreferredPaymentMode", "Gender",
    "PreferedOrderCat", "MaritalStatus",
]
NUMERIC_COLS = [
    "Tenure", "CityTier", "WarehouseToHome", "HourSpendOnApp",
    "NumberOfDeviceRegistered", "SatisfactionScore", "NumberOfAddress",
    "Complain", "OrderAmountHikeFromlastYear", "CouponUsed", "OrderCount",
    "DaySinceLastOrder", "CashbackAmount", "sentiment_score",
]
TARGET_COL = "Churn"

encoders = {}
model_df = df.copy()
for col in CATEGORICAL_COLS:
    le = LabelEncoder()
    model_df[col] = le.fit_transform(model_df[col].astype(str))
    encoders[col] = le

FEATURE_COLS = NUMERIC_COLS + CATEGORICAL_COLS
X = model_df[FEATURE_COLS]
y = model_df[TARGET_COL]

print("Features:", FEATURE_COLS)
print("Class balance:\n", y.value_counts(normalize=True))

## 3. Train / test split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)

## 4. Train models

`Churn` is imbalanced (~17% churners) — both models use class weighting so the minority (churn) class isn't ignored.

In [ ]:
from xgboost import XGBClassifier

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42,
)
xgb_model.fit(X_train, y_train)
print("XGBoost trained.")

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=400,
    max_depth=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
rf_model.fit(X_train, y_train)
print("Random Forest trained.")

## 5. Evaluate and compare

In [ ]:
from sklearn.metrics import (
    classification_report, roc_auc_score, confusion_matrix, f1_score
)

def evaluate(name, model):
    proba = model.predict_proba(X_test)[:, 1]
    pred = model.predict(X_test)
    print(f"--- {name} ---")
    print(classification_report(y_test, pred, digits=3))
    print("ROC-AUC:", round(roc_auc_score(y_test, proba), 4))
    print("Confusion matrix:\n", confusion_matrix(y_test, pred))
    print()
    return {"name": name, "f1": f1_score(y_test, pred), "roc_auc": roc_auc_score(y_test, proba)}

results = [evaluate("XGBoost", xgb_model), evaluate("Random Forest", rf_model)]
results_df = pd.DataFrame(results).sort_values("roc_auc", ascending=False)
results_df

In [ ]:
# Pick the best-performing model by ROC-AUC as the primary model for SHAP + serving.
# Both models are still saved to disk in section 9 for the backend to choose from.
best_name = results_df.iloc[0]["name"]
best_model = xgb_model if best_name == "XGBoost" else rf_model
print("Best model:", best_name)

## 6. SHAP — churn reasons

Uses `shap.TreeExplainer`, which works natively with both XGBoost and Random Forest.

In [ ]:
import shap

explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test)

# For RandomForestClassifier, shap_values is a list [class0, class1]; normalise to the churn-class array.
if isinstance(shap_values, list):
    shap_values_churn = shap_values[1]
else:
    shap_values_churn = shap_values

shap.summary_plot(shap_values_churn, X_test, feature_names=FEATURE_COLS)

In [ ]:
FEATURE_LABELS = {
    "Tenure": "short customer tenure",
    "CityTier": "city tier",
    "WarehouseToHome": "long delivery distance",
    "HourSpendOnApp": "low app engagement",
    "NumberOfDeviceRegistered": "few registered devices",
    "SatisfactionScore": "low satisfaction score",
    "NumberOfAddress": "few saved addresses",
    "Complain": "recent complaint filed",
    "OrderAmountHikeFromlastYear": "reduced spend vs. last year",
    "CouponUsed": "low coupon usage",
    "OrderCount": "fewer orders placed",
    "DaySinceLastOrder": "long time since last order",
    "CashbackAmount": "low cashback earned",
    "sentiment_score": "negative review sentiment",
    "PreferredLoginDevice": "login device pattern",
    "PreferredPaymentMode": "payment mode pattern",
    "Gender": "gender",
    "PreferedOrderCat": "preferred order category",
    "MaritalStatus": "marital status",
}

def top_reasons_for_row(row_idx, n=3):
    """Return the top-n SHAP-driven reasons pushing this customer toward churn."""
    row_shap = shap_values_churn[row_idx]
    row_features = X_test.iloc[row_idx]
    # only keep features pushing risk UP (positive shap value = pushes toward churn)
    contributions = sorted(
        zip(FEATURE_COLS, row_shap, row_features.values),
        key=lambda t: t[1],
        reverse=True,
    )
    reasons = []
    for feat, shap_val, feat_val in contributions:
        if shap_val <= 0:
            continue
        reasons.append(FEATURE_LABELS.get(feat, feat))
        if len(reasons) == n:
            break
    return reasons

# quick sanity check on one row
print(top_reasons_for_row(0))

## 7. Flag high-risk customers + generate SHAP reasons for all of them

In [ ]:
RISK_THRESHOLD = 0.6

test_customer_ids = df.loc[X_test.index, "CustomerID"].values
churn_proba = best_model.predict_proba(X_test)[:, 1]

results_table = pd.DataFrame({
    "CustomerID": test_customer_ids,
    "churn_probability": churn_proba,
}).reset_index(drop=True)

results_table["risk_level"] = np.where(
    results_table["churn_probability"] >= RISK_THRESHOLD, "high",
    np.where(results_table["churn_probability"] >= 0.35, "medium", "low")
)

reasons_list = [top_reasons_for_row(i) for i in range(len(X_test))]
results_table["top_reasons"] = reasons_list

print(results_table["risk_level"].value_counts())
results_table.sort_values("churn_probability", ascending=False).head(10)

## 8. AI-generated retention messages (Gemini)

For every **high-risk** customer, ask Gemini to write a short SMS-length retention message referencing their actual churn reasons, plus a suggested discount.

In [ ]:
from google import genai
from google.genai import types

try:
    from google.colab import userdata
    api_key = userdata.get("GEMINI_API_KEY")
except Exception:
    api_key = None

if not api_key:
    from getpass import getpass
    api_key = getpass("Enter your Gemini API key: ")

client = genai.Client(api_key=api_key)
MODEL_NAME = "gemini-3.5-flash-lite"
print("Gemini client ready.")

In [ ]:
import json
import time
import random

RETENTION_PROMPT = """
You are a customer retention copywriter for an e-commerce store.
A customer is at high risk of churning for these reasons: {reasons}.
Their churn probability is {prob:.0%}.

Write a short, warm, personalised SMS (max 160 characters) that:
- Acknowledges their experience without sounding robotic
- Offers a specific discount (pick a sensible % between 10-25 based on churn risk)
- Includes a clear call to action

Return ONLY compact JSON, no markdown:
{{"sms_text": "<the sms, <=160 chars>", "discount_percent": <int>}}
"""

def generate_retention_message(reasons, prob, retries=5):
    reasons_str = ", ".join(reasons) if reasons else "general disengagement"

    for attempt in range(retries):
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=RETENTION_PROMPT.format(reasons=reasons_str, prob=prob),
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    temperature=0.7,
                ),
            )
            data = json.loads(response.text)
            return data["sms_text"][:160], int(data["discount_percent"])

        except Exception as e:
            error = str(e)

            if "503" in error or "UNAVAILABLE" in error:
                wait_time = min(60, (2 ** attempt) + random.uniform(0, 2))
                print(f"\nGemini temporarily unavailable. Retry {attempt + 1}/{retries} after {wait_time:.1f}s...")
                time.sleep(wait_time)
                continue

            if "429" in error or "RESOURCE_EXHAUSTED" in error:
                wait_time = min(120, 5 * (2 ** attempt))
                print(f"\nRate limit reached. Waiting {wait_time}s...")
                time.sleep(wait_time)
                continue

            if attempt == retries - 1:
                print("Retention message failed, using fallback:", error)
                return "We miss you! Enjoy 15% off your next order as a thank-you for being our customer.", 15

            time.sleep(2)

    return "We miss you! Enjoy 15% off your next order as a thank-you for being our customer.", 15

high_risk = results_table[results_table["risk_level"] == "high"].copy()
print(f"{len(high_risk)} high-risk customers out of {len(results_table)} in the test set.")

sms_texts, discounts = [], []
for _, row in high_risk.iterrows():
    sms, disc = generate_retention_message(row["top_reasons"], row["churn_probability"])
    sms_texts.append(sms)
    discounts.append(disc)
    time.sleep(1)

high_risk["retention_sms"] = sms_texts
high_risk["discount_percent"] = discounts
high_risk[["CustomerID", "churn_probability", "top_reasons", "retention_sms", "discount_percent"]].head(10)

> Actually **sending** these messages over SMS is a backend concern (SMS gateway integration lives in the FastAPI service, not this notebook). This notebook only produces the message text + discount, ready to be picked up by that service.

## 9. Save all artifacts for the backend

In [ ]:
import json
import joblib

ARTIFACT_DIR = os.path.join(SAVE_DIR, "artifacts")
os.makedirs(ARTIFACT_DIR, exist_ok=True)

joblib.dump(xgb_model, os.path.join(ARTIFACT_DIR, "xgb_churn_model.joblib"))
joblib.dump(rf_model, os.path.join(ARTIFACT_DIR, "rf_churn_model.joblib"))
joblib.dump(encoders, os.path.join(ARTIFACT_DIR, "label_encoders.joblib"))
joblib.dump(FEATURE_COLS, os.path.join(ARTIFACT_DIR, "feature_columns.joblib"))
joblib.dump(explainer, os.path.join(ARTIFACT_DIR, "shap_explainer.joblib"))

with open(os.path.join(ARTIFACT_DIR, "model_comparison.json"), "w") as f:
    json.dump(results_df.to_dict(orient="records"), f, indent=2)

results_table.to_csv(os.path.join(ARTIFACT_DIR, "churn_predictions_test_set.csv"), index=False)
high_risk.to_csv(os.path.join(ARTIFACT_DIR, "high_risk_retention_messages.csv"), index=False)

print("Artifacts saved to:", ARTIFACT_DIR)
!ls -la {ARTIFACT_DIR}

---
## Done — ML part summary

- **Data:** merged churn dataset + Gemini-scored review sentiment (`sentiment_score` feature).
- **Models:** XGBoost and Random Forest trained, compared, best one selected by ROC-AUC.
- **Explainability:** SHAP gives global + per-customer churn reasons.
- **Retention:** Gemini auto-writes a personalised SMS + discount for every high-risk customer.
- **Artifacts saved to Drive** (`churn_prediction_project/artifacts/`): both trained models, encoders, feature list, SHAP explainer, and result CSVs.

**Next steps (outside this notebook, not started yet):**
1. FastAPI backend loads `xgb_churn_model.joblib` / `rf_churn_model.joblib` + `label_encoders.joblib` + `shap_explainer.joblib` to serve `/predict` and `/churn-reasons` endpoints.
2. Backend wires the SMS gateway to actually send `retention_sms` to high-risk customers.
3. React frontend calls the FastAPI backend to display churn probability, SHAP reasons, and retention offers per customer.